# Faz 3 · B3.1 — Colab ortamı

**Sahip:** Kişi B · **Model:** Qwen2.5-1.5B-Instruct + LoRA (Unsloth)

Bu defter eğitim yapmaz. Eğitimin **koşabileceği ortamı kurar ve kanıtlar.**
Boru hattı provası B3.2'de, gerçek eğitim B3.3'te.

---

## Ücretsiz Colab'ın üç kısıtı

Bunlar aksaklık değil, **tasarımı belirleyen şartlar**:

| kısıt | sonucu |
|---|---|
| Tek T4 GPU | ~55k örnek için 4-7 saat |
| Oturum ~90 dk hareketsizlikte, ~12 saatte kesin kopar | 4-7 saatlik tek koşu **büyük olasılıkla yarıda kesilir** |
| `/content` uçucu | Oturum kapanınca dosyalar gider |

Bu yüzden eğitim **tek koşu değil, kaldığı yerden devam edebilen** bir süreç
olarak kurulacak ve her şey **Drive'da** duracak. Tercih değil, zorunluluk.


## 1. GPU var mı

`Runtime → Change runtime type → T4 GPU` seçili olmalı.


In [ ]:
!nvidia-smi


In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4 GPU'

ad = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU  : {ad}')
print(f'VRAM : {vram:.1f} GB')
if 'T4' not in ad:
    print(f'{chr(10)}NOT: T4 bekleniyordu, {ad} bulundu. Sureler farkli cikar.')


## 2. Drive

Her Colab oturumu sıfırdan başlar; bu hücre **her açılışta** çalıştırılmalı.
Açılan pencereden Google hesabını seç ve izin ver.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Klasör düzeni

```
/content/drive/MyDrive/codifya/
  veri/        <- Kisi A'nin JSONL dosyalari (BIR KEZ yuklenir)
  checkpoint/  <- egitim ara kayitlari (oturum kopunca buradan devam)
  cikti/       <- merge edilmis model + GGUF
```

> ⚠️ **Veriyi her oturumda yeniden yükleme.** 76 MB'lık `gerekce_train.jsonl`
> her oturumda 10-20 dakika yakar ve kopma riskini artırır. Drive'a bir kez koy.


In [ ]:
from pathlib import Path

KOK = Path('/content/drive/MyDrive/codifya')
VERI = KOK / 'veri'
CHECKPOINT = KOK / 'checkpoint'
CIKTI = KOK / 'cikti'

for d in (VERI, CHECKPOINT, CIKTI):
    d.mkdir(parents=True, exist_ok=True)
    print(f'{d!s:45s} hazir')


## 4. Veri yerinde mi

Dosyalar Drive'daki `codifya/veri/` klasörüne elle yüklenir (Drive arayüzünden
sürükle-bırak). Aşağıdaki hücre eksik olanı söyler.


In [ ]:
BEKLENEN = {
    'gerekce_train.jsonl': 40293,
    'gerekce_val.jsonl': None,
    'gerekce_test.jsonl': None,
    'router_train.jsonl': 43798,
    'router_val.jsonl': None,
    'router_test.jsonl': None,
    'golden_set_aday.jsonl': None,
}

eksik = []
for ad_, beklenen in BEKLENEN.items():
    yol = VERI / ad_
    if not yol.exists():
        eksik.append(ad_)
        print(f'  YOK  {ad_}')
        continue
    mb = yol.stat().st_size / 1024**2
    with yol.open(encoding='utf-8') as f:
        n = sum(1 for _ in f)
    uyari = ''
    if beklenen and n != beklenen:
        uyari = f'  <-- {beklenen} bekleniyordu'
    print(f'  VAR  {ad_:24s} {mb:7.1f} MB  {n:6d} satir{uyari}')

if eksik:
    raise SystemExit(f'Eksik dosya: {eksik} -- Drive/codifya/veri/ klasorune yukle.')
print('Veri tamam.')


## 5. Unsloth

Kurulum 2-3 dakika sürer ve **her oturumda tekrarlanır** (Colab sıfırlanıyor).


In [ ]:
!pip install -q unsloth


In [ ]:
from unsloth import FastLanguageModel

print('unsloth hazir:', FastLanguageModel)


## 6. ⭐ İstem biçimi — eğitimden ÖNCE karara bağlanmalı

B3.1'in en kritik parçası. Yanlış karar 5 saatlik eğitimi çöpe atar.

### Bulunan çelişki

```
Egitim verisi (Kisi A) : gerekcelerin %100'unde urun adi AYNEN geciyor
B2.4 istemi   (Kisi B) : urun adini BILEREK koymuyor
```

B2.4'te adı çıkarmıştım çünkü **taban model** Türkçe özel adları bozuyordu:
`Astar Boya` → *starboy*, `İzocam` → *isyancı yalıtım levhası*. Guard bunu
yakalayamaz — uydurulan şey sayı değil.

### Neden iki biçim uyuşmak zorunda

Eğitimde istem ile hedef arasındaki ilişki neyse, model onu öğrenir.

- **İsteme ad koymaz, hedefte ad varsa** → model *yoktan ad uydurmayı* öğrenir.
  "starboy" sorununu ağırlıklara işler. **En kötü seçenek.**
- **İsteme ad koyar, hedefte ad varsa** → model *adı kopyalamayı* öğrenir.
  Taban model beceremiyordu çünkü hiç öğretilmemişti; 40 bin örnekle
  öğretilirse becermesi beklenir.

### Karar: ad isteme KONACAK

Melih'in 40.293 örneği yeniden etiketlenmeden kullanılabilsin diye.
Kopyalama davranışı **B3.2'de 100 örnekle sınanacak** — tam eğitimden önce, ucuz.

B3.2'de model yine ad bozarsa geri dönüş planı: hedeflerden adları ayıklayıp
ad içermeyen bir sürüm üretmek. Bir kerelik dönüşüm; veriyi yeniden üretmek
gerekmez.

### İkinci fark: eğitilmiş modelin istemi KISA

B2.4'ün istemi taban model için yazılmıştı; içinde kurallar bloğu ve few-shot
örnek var. Eğitilmiş modelde ikisi de gereksiz — davranış ağırlıklara işlenmiş
olacak. Kısa istem = az token = hızlı üretim.


In [ ]:
import json

# Egitim istemi. B2.4'teki app/llm/explain.py::istem_kur ile ayni mantik,
# iki bilincli fark: (1) urun adi VAR, (2) kurallar/few-shot YOK.

ETIKETLER = {
    'ort_gunluk_talep': 'gunluk ortalama talep (adet)',
    'tedarik_suresi_gun': 'tedarik suresi (gun)',
    'eldeki_stok': 'eldeki stok (adet)',
    'son_hareket_gun_once': 'son hareketten bu yana gecen gun',
    'birim_maliyet_tl': 'birim maliyet (TL)',
}


def tr_sayi(d):
    d = float(d)
    if d.is_integer():
        return f'{int(d):,}'.replace(',', '.')
    return f'{d:,.2f}'.replace(',', '~').replace('.', ',').replace('~', '.')


def istem_kur(kayit):
    o = kayit['ozellikler']
    satirlar = ['urun: ' + o['sku_adi'], 'karar: ' + kayit['karar_tipi']]
    for alan, etiket in ETIKETLER.items():
        deger = o.get(alan)
        if isinstance(deger, (int, float)) and not isinstance(deger, bool):
            satirlar.append(etiket + ': ' + tr_sayi(deger))
    return 'VERILER:' + chr(10) + chr(10).join(satirlar) + chr(10) * 2 + 'GEREKCE:'


with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    ornek = json.loads(f.readline())

print(istem_kur(ornek))
print()
print('--- HEDEF ---')
print(ornek['gerekce_metni'])


## 7. Bitti sayılır — üç ölçüt

Üçü de yeşilse B3.2'ye geçilir.


In [ ]:
olcutler = []

try:
    olcutler.append((torch.cuda.is_available(), 'GPU baglandi: ' + torch.cuda.get_device_name(0)))
except Exception as e:
    olcutler.append((False, 'GPU yok: ' + str(e)))

try:
    from unsloth import FastLanguageModel

    olcutler.append((KOK.exists(), 'Drive bagli: ' + str(KOK)))
    olcutler.append((True, 'unsloth import edildi'))
except Exception as e:
    olcutler.append((False, 'unsloth: ' + str(e)))

try:
    with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
        k = json.loads(f.readline())
    olcutler.append((bool(k.get('gerekce_metni')), 'veri/ okunabiliyor'))
except Exception as e:
    olcutler.append((False, 'veri okunamadi: ' + str(e)))

print('=' * 60)
for ok, mesaj in olcutler:
    print('  ' + ('GECTI' if ok else 'KALDI') + '  ' + mesaj)
print('=' * 60)
print('B3.1 TAMAM' if all(o for o, _ in olcutler) else 'B3.1 EKSIK')
